# Week 04 - Data Structures for Image Analysis

**MCTE 4323 / MCTA 4364 Machine Vision**

### Learning objectives
By the end of this lab you will be able to:
- Treat an image as a NumPy array and extract regions of interest (ROI).
- Build **Gaussian** and **Laplacian** image pyramids.
- Represent connected regions as **contours** and use their **topological hierarchy**.
- Implement a simple **quadtree** decomposition.

### Why data structures matter
A raw pixel array is rarely the final representation. Vision algorithms operate on **pyramids** (multi-scale), **contours/chains** (boundaries), **topological** structures (what contains what) and **quadtrees** (adaptive regions). Choosing the right structure can make an algorithm orders of magnitude faster.

## 1. Setup

In [ ]:
import os
if not os.path.isdir("MCTA-4364-Machine-Vision"):
    !git clone https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git
%cd MCTA-4364-Machine-Vision
!pip -q install opencv-python numpy matplotlib

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show(*images, titles=None, cmap=None):
    titles = titles or [""] * len(images)
    plt.figure(figsize=(5 * len(images), 5))
    for i, img in enumerate(images):
        plt.subplot(1, len(images), i + 1)
        plt.imshow(img, cmap=cmap or (None if img.ndim == 3 else "gray"))
        plt.title(titles[i]); plt.axis("off")
    plt.tight_layout(); plt.show()

img = cv2.imread("resources/images/hasan.jpg")
print("Shape:", img.shape)

## 2. Guided example - the image as an array and ROIs
An ROI is just NumPy slicing: `img[y0:y1, x0:x1]`. Remember the order is **rows (y) then columns (x)**.

In [ ]:
h, w = img.shape[:2]
roi = img[h//4:3*h//4, w//4:3*w//4].copy()

# Draw a box on a copy to show the ROI"s location
annotated = img.copy()
cv2.rectangle(annotated, (w//4, h//4), (3*w//4, 3*h//4), (0, 0, 255), 4)
show(annotated, roi, titles=["ROI location", "Extracted ROI"])

## 3. Guided example - image pyramids
Each `pyrDown` halves the resolution (and doubles the scale). The **Gaussian pyramid** smooths and downsamples; the **Laplacian pyramid** stores the detail lost at each level (useful for compression and blending).

In [ ]:
# Gaussian pyramid
g = img.copy()
gaussian = [g]
for _ in range(3):
    g = cv2.pyrDown(g)
    gaussian.append(g)
print("Gaussian levels:", [lvl.shape[:2] for lvl in gaussian])
show(*gaussian, titles=[f"L{i} {l.shape[1]}x{l.shape[0]}" for i, l in enumerate(gaussian)])

In [ ]:
# Laplacian pyramid: level_i = Gaussian_i - pyrUp(Gaussian_{i+1})
laplacian = []
for i in range(len(gaussian) - 1):
    up = cv2.pyrUp(gaussian[i + 1], dstsize=(gaussian[i].shape[1], gaussian[i].shape[0]))
    lap = cv2.subtract(gaussian[i], up)
    laplacian.append(lap)

# Show the detail of the first Laplacian level (add 128 for visibility)
vis_lap = cv2.add(laplacian[1], 128)
show(cv2.cvtColor(vis_lap, cv2.COLOR_BGR2RGB), titles=["Laplacian level 1 (detail)"])

## 4. Guided example - contours and topological hierarchy
`cv2.findContours` returns boundaries plus a **hierarchy** `[next, previous, first_child, parent]` that encodes the containment tree (topological structure).

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
_, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))

contours, hierarchy = cv2.findContours(binary, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
print("Number of contours:", len(contours))

vis = cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)
cv2.drawContours(vis, contours, -1, (0, 0, 255), 2)
show(binary, vis, titles=["Binary image", "Contours"])

In [ ]:
# Report area, perimeter and bounding box for the largest contours
contours_sorted = sorted(contours, key=cv2.contourArea, reverse=True)[:5]
for i, c in enumerate(contours_sorted):
    area = cv2.contourArea(c)
    peri = cv2.arcLength(c, True)
    x, y, bw, bh = cv2.boundingRect(c)
    print(f"#{i}: area={area:8.1f}  perimeter={peri:8.1f}  bbox=({x},{y},{bw},{bh})")

## 5. Guided example - quadtree decomposition
A **quadtree** recursively splits a region into four children while it is "too complex". Here complexity = intensity variance. Quadtrees give compact, adaptive representations of non-uniform images.

In [ ]:
def quadtree(im, x, y, size, threshold, depth=0):
    """Return list of leaf blocks (x, y, size) that are 'uniform enough'."""
    block = im[y:y + size, x:x + size]
    if size <= 1 or block.var() < threshold:
        return [(x, y, size)]
    half = size // 2
    leaves = []
    for dy in (0, half):
        for dx in (0, half):
            leaves += quadtree(im, x + dx, y + dy, half, threshold, depth + 1)
    return leaves

small = cv2.cvtColor(cv2.resize(gray, (256, 256)), cv2.COLOR_GRAY2BGR)
leaves = quadtree(gray[:256, :256], 0, 0, 256, threshold=200)
for x, y, size in leaves:
    cv2.rectangle(small, (x, y), (x + size, y + size), (0, 0, 255), 1)
print("Number of leaf blocks:", len(leaves))
show(small, titles=["Quadtree decomposition (var threshold = 200)"])

## 6. Exercise (complete the code)

1. **Reconstruct** the image from the first two Laplacian levels and the smallest Gaussian level (standard Laplacian reconstruction).
2. Compare it with the original - it should look almost identical.

*Hint:* start from the smallest Gaussian and repeatedly `pyrUp` and add the matching Laplacian level (`cv2.add`).

In [ ]:
# TODO: reconstruct from Gaussian + Laplacian pyramids


## 7. Challenge (independent)

Change the quadtree `threshold` to 50, 500 and 2000 and observe the number of leaves and the visual granularity. Then write one sentence describing the trade-off between **compression** (fewer leaves) and **fidelity** (more leaves).

In [ ]:
# Your code here


## 8. Reflection
1. Why is a pyramid useful for detecting objects at different scales?
2. In the contour hierarchy, what do `first_child` and `parent` represent for a hole inside an object?
3. Give an engineering example where a quadtree is more efficient than storing raw pixels.